In [3]:
"""
Práctica 5: Naïve Bayes
Escuela Superior de Cómputo
Profesora: Consuelo Varinia García Mendoza
"""

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, KFold
from sklearn.naive_bayes import GaussianNB, MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# ==================== FUNCIONES AUXILIARES ====================

def realizar_validacion_cruzada(X_train, y_train, modelo, k=3):
    """
    Realiza validación cruzada con k pliegues y retorna los accuracy de cada pliegue
    """
    kfold = KFold(n_splits=k, shuffle=True, random_state=0)
    accuracies = []
    
    for fold, (train_idx, val_idx) in enumerate(kfold.split(X_train), 1):
        # Dividir datos
        X_fold_train, X_fold_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        
        # Entrenar y predecir
        modelo.fit(X_fold_train, y_fold_train)
        y_pred = modelo.predict(X_fold_val)
        
        # Calcular accuracy
        acc = accuracy_score(y_fold_val, y_pred)
        accuracies.append(acc)
        
    return accuracies

def imprimir_tabla_validacion(resultados, nombre_dataset):
    """
    Imprime la tabla 1 con los resultados de validación cruzada
    """
    print(f"\n{'='*70}")
    print(f"TABLA 1: Resultados de Validación Cruzada - {nombre_dataset}")
    print(f"{'='*70}")
    print(f"{'Distribución':<20} {'Pliegue':<10} {'Accuracy':<15}")
    print(f"{'-'*70}")
    
    for dist_name, pliegues in resultados.items():
        for i, acc in enumerate(pliegues, 1):
            pliegue_txt = f"Pliegue {i}"
            print(f"{dist_name:<20} {pliegue_txt:<10} {acc:.6f}")
        
        promedio = np.mean(pliegues)
        print(f"{'':<20} {'Promedio':<10} {promedio:.6f}")
        print(f"{'-'*70}")

def evaluar_modelo_final(modelo, X_train, y_train, X_test, y_test, nombre_modelo, nombre_dataset):
    """
    Entrena el modelo final y genera la matriz de confusión y reporte de clasificación
    """
    # Entrenar con todos los datos de entrenamiento
    modelo.fit(X_train, y_train)
    
    # Predecir en conjunto de prueba
    y_pred = modelo.predict(X_test)
    
    # Calcular accuracy
    acc = accuracy_score(y_test, y_pred)
    
    print(f"\n{'='*70}")
    print(f"RESULTADOS FINALES - {nombre_dataset} ({nombre_modelo})")
    print(f"{'='*70}")
    print(f"Accuracy en conjunto de prueba: {acc:.6f}")
    
    # Reporte de clasificación
    print("\n--- Reporte de Clasificación ---")
    print(classification_report(y_test, y_pred))
    
    # Matriz de confusión
    cm = confusion_matrix(y_test, y_pred)
    print("\n--- Matriz de Confusión ---")
    print(cm)
    
    # Visualizar matriz de confusión
    fig, ax = plt.subplots(figsize=(8, 6))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot(ax=ax, cmap='Blues', values_format='d')
    plt.title(f'Matriz de Confusión - {nombre_dataset}\n{nombre_modelo}')
    plt.tight_layout()
    plt.savefig(f'/mnt/user-data/outputs/confusion_matrix_{nombre_dataset}_{nombre_modelo}.png', dpi=300, bbox_inches='tight')
    print(f"\nMatriz de confusión guardada como: confusion_matrix_{nombre_dataset}_{nombre_modelo}.png")
    plt.close()
    
    return acc

# ==================== PROCESAMIENTO IRIS.CSV ====================

def procesar_iris():
    """
    Procesa el dataset iris.csv
    """
    print("\n" + "="*70)
    print("PROCESANDO DATASET: iris.csv")
    print("="*70)
    
    try:
        # Cargar datos
        df_iris = pd.read_csv('iris.csv')
        print(f"Dataset cargado: {df_iris.shape[0]} instancias, {df_iris.shape[1]} columnas")
        
        # Separar características y clase
        X = df_iris.iloc[:, :-1]  # Primeras 4 columnas
        y = df_iris.iloc[:, -1]    # Última columna (clase)
        
        # Mezclar y dividir datos (70% entrenamiento, 30% prueba)
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.3, random_state=0, shuffle=True
        )
        
        print(f"Conjunto de entrenamiento: {X_train.shape[0]} instancias")
        print(f"Conjunto de prueba: {X_test.shape[0]} instancias")
        
        # ========== VALIDACIÓN CRUZADA ==========
        resultados_cv = {}
        
        # Distribución Normal (GaussianNB)
        print("\n--- Validación Cruzada con GaussianNB (Normal) ---")
        modelo_gaussian = GaussianNB()
        acc_gaussian = realizar_validacion_cruzada(X_train, y_train, modelo_gaussian, k=3)
        resultados_cv['Normal'] = acc_gaussian
        
        # Distribución Multinomial (MultinomialNB)
        # Para MultinomialNB necesitamos valores no negativos
        # Si hay valores negativos, hacer scaling
        X_train_pos = X_train.copy()
        X_test_pos = X_test.copy()
        
        if (X_train_pos < 0).any().any():
            min_val = X_train_pos.min().min()
            X_train_pos = X_train_pos - min_val
            X_test_pos = X_test_pos - min_val
        
        print("\n--- Validación Cruzada con MultinomialNB ---")
        modelo_multinomial = MultinomialNB()
        acc_multinomial = realizar_validacion_cruzada(X_train_pos, y_train, modelo_multinomial, k=3)
        resultados_cv['Multinomial'] = acc_multinomial
        
        # Imprimir Tabla 1
        imprimir_tabla_validacion(resultados_cv, "iris.csv")
        
        # ========== SELECCIONAR MEJOR MODELO ==========
        promedio_gaussian = np.mean(acc_gaussian)
        promedio_multinomial = np.mean(acc_multinomial)
        
        print(f"\nPromedio Accuracy GaussianNB: {promedio_gaussian:.6f}")
        print(f"Promedio Accuracy MultinomialNB: {promedio_multinomial:.6f}")
        
        if promedio_gaussian >= promedio_multinomial:
            print("\n>>> Mejor modelo: GaussianNB (Normal)")
            mejor_modelo = GaussianNB()
            mejor_nombre = "GaussianNB"
            X_train_final = X_train
            X_test_final = X_test
        else:
            print("\n>>> Mejor modelo: MultinomialNB")
            mejor_modelo = MultinomialNB()
            mejor_nombre = "MultinomialNB"
            X_train_final = X_train_pos
            X_test_final = X_test_pos
        
        # ========== EVALUACIÓN FINAL ==========
        acc_final = evaluar_modelo_final(
            mejor_modelo, X_train_final, y_train, X_test_final, y_test,
            mejor_nombre, "iris"
        )
        
        return {
            'dataset': 'iris.csv',
            'distribucion': mejor_nombre,
            'accuracy': acc_final
        }
        
    except FileNotFoundError:
        print("\n⚠️  ADVERTENCIA: No se encontró el archivo iris.csv")
        print("Por favor, sube el archivo a /mnt/user-data/uploads/iris.csv")
        return None
    except Exception as e:
        print(f"\n❌ ERROR al procesar iris.csv: {str(e)}")
        return None

# ==================== PROCESAMIENTO EMAILS.CSV ====================

def procesar_emails():
    """
    Procesa el dataset emails.csv
    """
    print("\n" + "="*70)
    print("PROCESANDO DATASET: emails.csv")
    print("="*70)
    
    try:
        # Cargar datos
        df_emails = pd.read_csv('emails.csv')
        print(f"Dataset cargado: {df_emails.shape[0]} instancias, {df_emails.shape[1]} columnas")
        
        # Separar características y clase
        # Primera columna es ID (se descarta)
        # Última columna es la clase (spam o no)
        # Columnas intermedias son las palabras
        X = df_emails.iloc[:, 1:-1]  # Columnas de palabras
        y = df_emails.iloc[:, -1]     # Última columna (spam)
        
        print(f"Características (palabras): {X.shape[1]}")
        
        # Mezclar y dividir datos (70% entrenamiento, 30% prueba)
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.3, random_state=0, shuffle=True
        )
        
        print(f"Conjunto de entrenamiento: {X_train.shape[0]} instancias")
        print(f"Conjunto de prueba: {X_test.shape[0]} instancias")
        
        # ========== VALIDACIÓN CRUZADA ==========
        resultados_cv = {}
        
        # Distribución Normal (GaussianNB)
        print("\n--- Validación Cruzada con GaussianNB (Normal) ---")
        modelo_gaussian = GaussianNB()
        acc_gaussian = realizar_validacion_cruzada(X_train, y_train, modelo_gaussian, k=3)
        resultados_cv['Normal'] = acc_gaussian
        
        # Distribución Multinomial (MultinomialNB)
        # MultinomialNB es ideal para datos de conteo de palabras
        print("\n--- Validación Cruzada con MultinomialNB ---")
        modelo_multinomial = MultinomialNB()
        acc_multinomial = realizar_validacion_cruzada(X_train, y_train, modelo_multinomial, k=3)
        resultados_cv['Multinomial'] = acc_multinomial
        
        # Imprimir Tabla 1
        imprimir_tabla_validacion(resultados_cv, "emails.csv")
        
        # ========== SELECCIONAR MEJOR MODELO ==========
        promedio_gaussian = np.mean(acc_gaussian)
        promedio_multinomial = np.mean(acc_multinomial)
        
        print(f"\nPromedio Accuracy GaussianNB: {promedio_gaussian:.6f}")
        print(f"Promedio Accuracy MultinomialNB: {promedio_multinomial:.6f}")
        
        if promedio_gaussian >= promedio_multinomial:
            print("\n>>> Mejor modelo: GaussianNB (Normal)")
            mejor_modelo = GaussianNB()
            mejor_nombre = "GaussianNB"
        else:
            print("\n>>> Mejor modelo: MultinomialNB")
            mejor_modelo = MultinomialNB()
            mejor_nombre = "MultinomialNB"
        
        # ========== EVALUACIÓN FINAL ==========
        acc_final = evaluar_modelo_final(
            mejor_modelo, X_train, y_train, X_test, y_test,
            mejor_nombre, "emails"
        )
        
        return {
            'dataset': 'emails.csv',
            'distribucion': mejor_nombre,
            'accuracy': acc_final
        }
        
    except FileNotFoundError:
        print("\n⚠️  ADVERTENCIA: No se encontró el archivo emails.csv")
        print("Por favor, sube el archivo a /mnt/user-data/uploads/emails.csv")
        return None
    except Exception as e:
        print(f"\n❌ ERROR al procesar emails.csv: {str(e)}")
        return None

# ==================== PROGRAMA PRINCIPAL ====================

def main():
    """
    Función principal que ejecuta todo el análisis
    """
    print("\n" + "="*70)
    print("PRÁCTICA 5: NAÏVE BAYES")
    print("Escuela Superior de Cómputo")
    print("="*70)
    
    # Crear directorio de salida
    import os
    os.makedirs('/mnt/user-data/outputs', exist_ok=True)
    
    # Procesar ambos datasets
    resultado_iris = procesar_iris()
    resultado_emails = procesar_emails()
    
    # ========== TABLA 2: RESUMEN FINAL ==========
    print("\n" + "="*70)
    print("TABLA 2: Resultados de las Pruebas Finales")
    print("="*70)
    print(f"{'Dataset':<20} {'Distribución':<20} {'Accuracy':<15}")
    print("-"*70)
    
    if resultado_iris:
        print(f"{resultado_iris['dataset']:<20} {resultado_iris['distribucion']:<20} {resultado_iris['accuracy']:.6f}")
    else:
        print(f"{'iris.csv':<20} {'N/A':<20} {'N/A':<15}")
    
    if resultado_emails:
        print(f"{resultado_emails['dataset']:<20} {resultado_emails['distribucion']:<20} {resultado_emails['accuracy']:.6f}")
    else:
        print(f"{'emails.csv':<20} {'N/A':<20} {'N/A':<15}")
    
    print("="*70)
    print("\n✅ Análisis completado")
    print("\nArchivos generados en /mnt/user-data/outputs:")
    print("  - confusion_matrix_iris_[modelo].png")
    print("  - confusion_matrix_emails_[modelo].png")

if __name__ == "__main__":
    main()


PRÁCTICA 5: NAÏVE BAYES
Escuela Superior de Cómputo

PROCESANDO DATASET: iris.csv
Dataset cargado: 150 instancias, 5 columnas
Conjunto de entrenamiento: 105 instancias
Conjunto de prueba: 45 instancias

--- Validación Cruzada con GaussianNB (Normal) ---

--- Validación Cruzada con MultinomialNB ---

TABLA 1: Resultados de Validación Cruzada - iris.csv
Distribución         Pliegue    Accuracy       
----------------------------------------------------------------------
Normal               Pliegue 1  0.971429
Normal               Pliegue 2  0.914286
Normal               Pliegue 3  0.942857
                     Promedio   0.942857
----------------------------------------------------------------------
Multinomial          Pliegue 1  0.857143
Multinomial          Pliegue 2  0.714286
Multinomial          Pliegue 3  0.600000
                     Promedio   0.723810
----------------------------------------------------------------------

Promedio Accuracy GaussianNB: 0.942857
Promedio Accurac